In [ ]:
# Verify cbam.py is working
import importlib
import sys
sys.path.insert(0, '/content/drive/MyDrive/brain_tumor_classification')

import src.cbam
importlib.reload(src.cbam)

from src.cbam import CBAMEfficientNetB3, get_cbam_optimizer
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = CBAMEfficientNetB3(num_classes=4, pretrained=True).to(device)
model.print_model_info()

dummy = torch.randn(4, 3, 224, 224).to(device)
with torch.no_grad():
    logits = model(dummy)

print(f"\nOutput shape: {logits.shape}")
print("CBAM model ready for training.")

  CBAMEfficientNetB3 — Model Summary
  Backbone       : EfficientNetB3 (pretrained)
  CBAM params    : 296,642
  Total params   : 11,387,374
  Trainable now  : 691,142
  Dropout        : 0.3
  Output classes : 4

Output shape: torch.Size([4, 4])
CBAM model ready for training.


In [ ]:
# Train CBAMEfficientNetB3
import os
import torch
import torch.nn as nn
from tqdm import tqdm
from sklearn.metrics import f1_score, accuracy_score

from src.cbam import CBAMEfficientNetB3, get_cbam_optimizer
from src.dataset import create_dataloaders
from utils.logger import get_logger, MetricLogger

device   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_DIR = '/content/drive/MyDrive/brain_tumor_classification/data'
EXP_DIR  = '/content/drive/MyDrive/brain_tumor_classification/experiments/cbam_efficientnet_b3'
os.makedirs(EXP_DIR, exist_ok=True)

# ── DataLoaders (same as all other experiments)
train_loader, val_loader, test_loader, info = create_dataloaders(
    data_dir   = DATA_DIR,
    image_size = 224,
    batch_size = 32,
    val_split  = 0.2,
    seed       = 42,
)
print(f"Train: {info['train_size']} | Val: {info['val_size']}")

# ── Model
model = CBAMEfficientNetB3(
    num_classes  = 4,
    dropout_rate = 0.3,
    pretrained   = True,
).to(device)

# ── Loss + AMP scaler
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
scaler = torch.amp.GradScaler('cuda')

# ── Logging
logger         = get_logger('cbam_efficientnet_b3', log_dir=EXP_DIR)
metric_tracker = MetricLogger(log_dir=EXP_DIR)

# ── Training settings (same 3-phase strategy)
EPOCHS           = 30
WARMUP_EPOCHS    = 5
FINETUNE_EPOCH   = 20
PATIENCE         = 7

best_val_f1            = 0.0
epochs_no_improve      = 0
optimizer              = None
scheduler              = None

logger.info("Starting CBAM EfficientNetB3 training...")

for epoch in range(1, EPOCHS + 1):

    # ── Phase transitions
    if epoch == 1:
        logger.info("PHASE 1 — backbone frozen, CBAM + head train")
        model.freeze_backbone()
        optimizer = get_cbam_optimizer(model, head_lr=1e-3, backbone_lr=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=WARMUP_EPOCHS, eta_min=1e-6)

    elif epoch == WARMUP_EPOCHS + 1:
        logger.info("PHASE 2 — top backbone blocks unfrozen")
        model.unfreeze_top_layers(num_blocks=2)
        optimizer = get_cbam_optimizer(model, head_lr=1e-4, backbone_lr=1e-5)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=FINETUNE_EPOCH - WARMUP_EPOCHS, eta_min=1e-6)

    elif epoch == FINETUNE_EPOCH + 1:
        logger.info("PHASE 3 — full model unfrozen")
        model.unfreeze_all()
        optimizer = get_cbam_optimizer(model, head_lr=1e-5, backbone_lr=1e-5)
        remaining = EPOCHS - FINETUNE_EPOCH
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=max(remaining, 1), eta_min=1e-6)

    # ── Training pass
    model.train()
    train_loss, train_preds, train_labels = 0.0, [], []

    pbar = tqdm(train_loader, desc=f"Epoch {epoch:02d} [Train]", leave=False)
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()

        with torch.amp.autocast('cuda'):
            logits = model(images)
            loss   = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item() * images.size(0)
        train_preds.extend(torch.argmax(logits, 1).cpu().numpy())
        train_labels.extend(labels.cpu().numpy())
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})

    train_loss /= len(train_loader.dataset)
    train_acc   = accuracy_score(train_labels, train_preds)
    train_f1    = f1_score(train_labels, train_preds, average='macro', zero_division=0)

    # ── Validation pass
    model.eval()
    val_loss, val_preds, val_labels = 0.0, [], []

    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch:02d} [Val]  ", leave=False):
            images, labels = images.to(device), labels.to(device)
            logits  = model(images)
            loss    = criterion(logits, labels)
            val_loss += loss.item() * images.size(0)
            val_preds.extend(torch.argmax(logits, 1).cpu().numpy())
            val_labels.extend(labels.cpu().numpy())

    val_loss /= len(val_loader.dataset)
    val_acc   = accuracy_score(val_labels, val_preds)
    val_f1    = f1_score(val_labels, val_preds, average='macro', zero_division=0)

    scheduler.step()

    metric_tracker.update(epoch, 'train', train_loss, train_acc, train_f1)
    metric_tracker.update(epoch, 'val',   val_loss,   val_acc,   val_f1)

    logger.info(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} F1: {train_f1:.4f} | "
        f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} F1: {val_f1:.4f}"
    )

    # ── Save best model
    if val_f1 > best_val_f1 + 1e-4:
        best_val_f1     = val_f1
        epochs_no_improve = 0
        torch.save(model.state_dict(), os.path.join(EXP_DIR, 'best_model.pth'))
        logger.info(f"  --> Best model saved (Val F1: {best_val_f1:.4f})")
    else:
        epochs_no_improve += 1
        logger.info(f"  No improvement. Patience: {epochs_no_improve}/{PATIENCE}")

    if epochs_no_improve >= PATIENCE:
        logger.info(f"Early stopping at epoch {epoch}.")
        break

metric_tracker.save()
logger.info(f"Training complete. Best Val F1: {best_val_f1:.4f}")
print(f"\nCBAM training complete. Best Val F1: {best_val_f1:.4f}")

Train: 4480 | Val: 1120
2026-04-07 05:14:43 | INFO     | Log file: /content/drive/MyDrive/brain_tumor_classification/experiments/cbam_efficientnet_b3/cbam_efficientnet_b3_20260407_051443.log


INFO:cbam_efficientnet_b3:Log file: /content/drive/MyDrive/brain_tumor_classification/experiments/cbam_efficientnet_b3/cbam_efficientnet_b3_20260407_051443.log


2026-04-07 05:14:43 | INFO     | Starting CBAM EfficientNetB3 training...


INFO:cbam_efficientnet_b3:Starting CBAM EfficientNetB3 training...


2026-04-07 05:14:43 | INFO     | PHASE 1 — backbone frozen, CBAM + head train


...

2026-04-07 05:46:53 | INFO     |   No improvement. Patience: 1/7


INFO:cbam_efficientnet_b3:  No improvement. Patience: 1/7


2026-04-07 05:46:53 | INFO     | Training complete. Best Val F1: 0.9387


INFO:cbam_efficientnet_b3:Training complete. Best Val F1: 0.9387



CBAM training complete. Best Val F1: 0.9387


In [ ]:
import yaml, os

# Manually save the config.yaml that the training cell skipped
cbam_exp_dir = '/content/drive/MyDrive/brain_tumor_classification/experiments/cbam_efficientnet_b3'
os.makedirs(cbam_exp_dir, exist_ok=True)

config_dict = {
    "name": "cbam_efficientnet_b3",
    "model": {
        "backbone":     "efficientnet_b3",
        "pretrained":   True,
        "num_classes":  4,
    },
    "regularization": {
        "dropout_rate": 0.3,
        "use_mixup":    False,
        "use_cutmix":   False,
    },
    "data": {
        "image_size":  224,
        "batch_size":  32,
        "val_split":   0.2,
        "seed":        42,
        "num_workers": 2,
    },
    "train": {
        "epochs":              30,
        "warmup_epochs":        5,
        "warmup_lr":          1e-3,
        "finetune_lr":        1e-4,
        "full_finetune_lr":   1e-5,
        "full_finetune_epoch": 20,
        "optimizer":          "adamw",
        "weight_decay":       1e-4,
        "scheduler":          "cosine",
        "lr_min":             1e-6,
        "label_smoothing":    0.1,
        "patience":            7,
        "use_amp":             True,
    },
}

config_path = os.path.join(cbam_exp_dir, "config.yaml")
with open(config_path, "w") as f:
    yaml.dump(config_dict, f, default_flow_style=False, sort_keys=False)

print(f"config.yaml saved → {config_path}")

# Verify best_model.pth also exists
model_path = os.path.join(cbam_exp_dir, "best_model.pth")
print(f"best_model.pth exists: {os.path.exists(model_path)}")

config.yaml saved → /content/drive/MyDrive/brain_tumor_classification/experiments/cbam_efficientnet_b3/config.yaml
best_model.pth exists: True


In [ ]:
# CBAM-aware evaluation function
import torch
import numpy as np
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report
from sklearn.preprocessing import label_binarize

from src.cbam import CBAMEfficientNetB3
from src.dataset import BrainTumorDataset, CLASS_NAMES
from src.transforms import get_transforms
from src.evaluate import compute_metrics, plot_confusion_matrix

def evaluate_cbam_experiment(
    experiment_name = 'cbam_efficientnet_b3',
    device          = None,
    data_dir        = '/content/drive/MyDrive/brain_tumor_classification/data',
    experiments_dir = '/content/drive/MyDrive/brain_tumor_classification/experiments',
    outputs_dir     = '/content/drive/MyDrive/brain_tumor_classification/outputs',
    plot_cm         = True,
):
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    exp_dir    = os.path.join(experiments_dir, experiment_name)
    model_path = os.path.join(exp_dir, 'best_model.pth')

    print(f"\n{'='*52}")
    print(f"  Evaluating: {experiment_name}")
    print(f"{'='*52}")
    print(f"  Backbone  : EfficientNetB3 + CBAM")
    print(f"  Device    : {device}")

    # ── Rebuild CBAM model and load weights
    model = CBAMEfficientNetB3(
        num_classes  = 4,
        dropout_rate = 0.3,
        pretrained   = False,
    )
    state_dict = torch.load(model_path, map_location=device)
    model.load_state_dict(state_dict)
    model = model.to(device)
    model.eval()

    # ── Test DataLoader
    test_transform = get_transforms(image_size=224, phase='test')
    test_dataset   = BrainTumorDataset(
        root_dir  = os.path.join(data_dir, 'Testing'),
        transform = test_transform,
    )
    test_loader = DataLoader(test_dataset, batch_size=32,
                             shuffle=False, num_workers=2)
    print(f"  Test set  : {len(test_dataset)} images")

    # ── Get predictions
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for images, labels in test_loader:
            images  = images.to(device)
            logits  = model(images)
            probs   = torch.softmax(logits, dim=1)
            preds   = torch.argmax(logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
            all_probs.extend(probs.cpu().numpy())

    all_probs = np.array(all_probs)

    # ── Compute metrics
    metrics = compute_metrics(all_preds, all_labels, all_probs)

    print(f"\n  Accuracy  : {metrics['accuracy']:.2f}%")
    print(f"  Macro F1  : {metrics['macro_f1']:.4f}")
    print(f"  Macro AUC : {metrics['macro_auc']}")
    print(f"\n  Per-class F1:")
    for cls in CLASS_NAMES:
        print(f"    {cls:<15} {metrics[f'f1_{cls}']:.4f}")
    print(f"\n  Classification Report:")
    print(metrics['report'])

    if plot_cm:
        cm_path = plot_confusion_matrix(
            all_labels, all_preds,
            experiment_name = experiment_name,
            outputs_dir     = outputs_dir,
        )
        print(f"  Confusion matrix saved → {cm_path}")

    metrics['experiment'] = experiment_name
    metrics['backbone']   = 'efficientnet_b3 + cbam'
    metrics['dropout']    = 0.3
    metrics['label_smooth'] = 0.1
    metrics['weight_decay'] = 1e-4
    return metrics

In [ ]:
# CBAM-aware evaluation function
import torch
import numpy as np
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report
from sklearn.preprocessing import label_binarize

from src.cbam import CBAMEfficientNetB3
from src.dataset import BrainTumorDataset, CLASS_NAMES
from src.transforms import get_transforms
from src.evaluate import compute_metrics, plot_confusion_matrix

def evaluate_cbam_experiment(
    experiment_name = 'cbam_efficientnet_b3',
    device          = None,
    data_dir        = '/content/drive/MyDrive/brain_tumor_classification/data',
    experiments_dir = '/content/drive/MyDrive/brain_tumor_classification/experiments',
    outputs_dir     = '/content/drive/MyDrive/brain_tumor_classification/outputs',
    plot_cm         = True,
):
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    exp_dir    = os.path.join(experiments_dir, experiment_name)
    model_path = os.path.join(exp_dir, 'best_model.pth')

    print(f"\n{'='*52}")
    print(f"  Evaluating: {experiment_name}")
    print(f"{'='*52}")
    print(f"  Backbone  : EfficientNetB3 + CBAM")
    print(f"  Device    : {device}")

    # ── Rebuild CBAM model and load weights
    model = CBAMEfficientNetB3(
        num_classes  = 4,
        dropout_rate = 0.3,
        pretrained   = False,
    )
    state_dict = torch.load(model_path, map_location=device)
    model.load_state_dict(state_dict)
    model = model.to(device)
    model.eval()

    # ── Test DataLoader
    test_transform = get_transforms(image_size=224, phase='test')
    test_dataset   = BrainTumorDataset(
        root_dir  = os.path.join(data_dir, 'Testing'),
        transform = test_transform,
    )
    test_loader = DataLoader(test_dataset, batch_size=32,
                             shuffle=False, num_workers=2)
    print(f"  Test set  : {len(test_dataset)} images")

    # ── Get predictions
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for images, labels in test_loader:
            images  = images.to(device)
            logits  = model(images)
            probs   = torch.softmax(logits, dim=1)
            preds   = torch.argmax(logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
            all_probs.extend(probs.cpu().numpy())

    all_probs = np.array(all_probs)

    # ── Compute metrics
    metrics = compute_metrics(all_preds, all_labels, all_probs)

    print(f"\n  Accuracy  : {metrics['accuracy']:.2f}%")
    print(f"  Macro F1  : {metrics['macro_f1']:.4f}")
    print(f"  Macro AUC : {metrics['macro_auc']}")
    print(f"\n  Per-class F1:")
    for cls in CLASS_NAMES:
        print(f"    {cls:<15} {metrics[f'f1_{cls}']:.4f}")
    print(f"\n  Classification Report:")
    print(metrics['report'])

    if plot_cm:
        cm_path = plot_confusion_matrix(
            all_labels, all_preds,
            experiment_name = experiment_name,
            outputs_dir     = outputs_dir,
        )
        print(f"  Confusion matrix saved → {cm_path}")

    metrics['experiment'] = experiment_name
    metrics['backbone']   = 'efficientnet_b3 + cbam'
    metrics['dropout']    = 0.3
    metrics['label_smooth'] = 0.1
    metrics['weight_decay'] = 1e-4
    return metrics

In [ ]:
# Now evaluate both models and compare
cbam_results = evaluate_cbam_experiment(device=device)

from src.evaluate import evaluate_experiment
base_results = evaluate_experiment(
    experiment_name = 'baseline_efficientnet_b3',
    device          = device,
    plot_cm         = False,
)

# ── Print comparison table
import pandas as pd

rows = []
for name, r in [("EfficientNetB3 (baseline)", base_results),
                ("EfficientNetB3 + CBAM (proposed)", cbam_results)]:
    rows.append({
        "Model":         name,
        "Accuracy (%)":  r["accuracy"],
        "Macro F1":      r["macro_f1"],
        "Macro AUC":     r["macro_auc"],
        "F1 Glioma":     r["f1_glioma"],
        "F1 Meningioma": r["f1_meningioma"],
        "F1 Notumor":    r["f1_notumor"],
        "F1 Pituitary":  r["f1_pituitary"],
    })

df = pd.DataFrame(rows)
print("\n" + "=" * 80)
print("  Phase 5 — CBAM vs Baseline (TEST SET)")
print("=" * 80)
print(df.to_string(index=False))
print("=" * 80)

print("\n  Per-class F1 change:")
for cls in ["glioma", "meningioma", "notumor", "pituitary"]:
    diff = cbam_results[f"f1_{cls}"] - base_results[f"f1_{cls}"]
    print(f"    {cls:<15} {diff:+.4f}")

print(f"\n  Macro F1 change : {cbam_results['macro_f1'] - base_results['macro_f1']:+.4f}")
print(f"  Accuracy change : {cbam_results['accuracy'] - base_results['accuracy']:+.2f}%")


  Evaluating: cbam_efficientnet_b3
  Backbone  : EfficientNetB3 + CBAM
  Device    : cuda
  Test set  : 1600 images

  Accuracy  : 89.06%
  Macro F1  : 0.8878
  Macro AUC : 0.9771

  Per-class F1:
    glioma          0.8235
    meningioma      0.8458
    notumor         0.9318
    pituitary       0.9502

  Classification Report:
              precision    recall  f1-score   support

      glioma       0.97      0.72      0.82       400
  meningioma       0.82      0.88      0.85       400
     notumor       0.88      0.99      0.93       400
   pituitary       0.92      0.98      0.95       400

    accuracy                           0.89      1600
   macro avg       0.90      0.89      0.89      1600
weighted avg       0.90      0.89      0.89      1600

  Confusion matrix saved → /content/drive/MyDrive/brain_tumor_classification/outputs/confusion_matrix_cbam_efficientnet_b3.png

  Evaluating: baseline_efficientnet_b3
  Backbone  : efficientnet_b3
  Device    : cuda
  Test set  : 160

In [ ]:
!pip install grad-cam

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 47.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for grad-cam: filename=grad_cam-1.5.5-py3-none-any.whl size=44286 sha256=8c6972b350b8716e9dd65e27ca5f0a154ffc757890d66ae370c34ab9883d153e
  Stored in directory: /root/.cache/pip/wheels/fb/3b/09/2afc520f3d69bc26ae6bd87416759c820a3f7d05c1a077bbf6
Successfully built grad-cam


In [ ]:
# ── Cell 1 (REPLACE the previous version with this)
import torch
import numpy as np
import cv2
import os
import matplotlib.pyplot as plt
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from src.gradcam import denormalize_image
from src.dataset import CLASS_NAMES

def generate_gradcam_fixed(model, target_layer, image_tensor, device=None):
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    model.eval()
    input_tensor = image_tensor.unsqueeze(0).to(device)

    # Get prediction first (no gradients needed)
    with torch.no_grad():
        logits     = model(input_tensor)
        probs      = torch.softmax(logits, dim=1)
        pred_class = torch.argmax(probs, dim=1).item()
        pred_prob  = probs[0, pred_class].item()

    # ── Temporarily unfreeze ALL parameters so gradients can flow
    # This is only for the CAM computation — we restore afterwards
    original_requires_grad = {}
    for name, param in model.named_parameters():
        original_requires_grad[name] = param.requires_grad
        param.requires_grad_(True)

    try:
        cam = GradCAM(model=model, target_layers=[target_layer])
        grayscale_cam = cam(input_tensor=input_tensor, targets=None)
        grayscale_cam = grayscale_cam[0]
    finally:
        # ── Always restore original requires_grad state
        for name, param in model.named_parameters():
            param.requires_grad_(original_requires_grad[name])

    original_image = denormalize_image(image_tensor)
    cam_image = show_cam_on_image(
        original_image,
        grayscale_cam,
        use_rgb      = True,
        colormap     = cv2.COLORMAP_JET,
        image_weight = 0.5,
    )
    cam_image = cam_image.astype(np.float32) / 255.0

    return cam_image, pred_class, pred_prob, grayscale_cam


def plot_comparison_fixed(model_a, layer_a, model_b, layer_b,
                          samples, label_a, label_b, device,
                          save_path=None):

    n = len(samples)
    fig, axes = plt.subplots(n, 3, figsize=(12, n * 3.5))
    if n == 1:
        axes = axes[None, :]

    fig.suptitle(f"Grad-CAM: {label_a}  vs  {label_b}",
                 fontsize=13, fontweight='bold', y=1.01)

    col_titles = ["Original MRI",
                  f"{label_a}\nGrad-CAM",
                  f"{label_b}\nGrad-CAM"]
    for col, ct in enumerate(col_titles):
        axes[0, col].set_title(ct, fontsize=10, fontweight='bold', pad=10)

    for row, (img_tensor, true_label, _) in enumerate(samples):
        true_name = CLASS_NAMES[true_label].capitalize()

        original = denormalize_image(img_tensor)
        axes[row, 0].imshow(original, cmap='gray')
        axes[row, 0].set_ylabel(f"True: {true_name}", fontsize=9,
                                fontweight='bold', rotation=0,
                                labelpad=70, va='center')
        axes[row, 0].axis('off')

        cam_a, pred_a, prob_a, _ = generate_gradcam_fixed(
            model_a, layer_a, img_tensor, device)
        color_a = '#2ca02c' if pred_a == true_label else '#d62728'
        axes[row, 1].imshow(cam_a)
        axes[row, 1].set_title(
            f"Pred: {CLASS_NAMES[pred_a].capitalize()} ({prob_a:.0%})",
            fontsize=9, color=color_a)
        axes[row, 1].axis('off')

        cam_b, pred_b, prob_b, _ = generate_gradcam_fixed(
            model_b, layer_b, img_tensor, device)
        color_b = '#2ca02c' if pred_b == true_label else '#d62728'
        axes[row, 2].imshow(cam_b)
        axes[row, 2].set_title(
            f"Pred: {CLASS_NAMES[pred_b].capitalize()} ({prob_b:.0%})",
            fontsize=9, color=color_b)
        axes[row, 2].axis('off')

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, bbox_inches='tight', dpi=150)
        plt.close()
        print(f"Saved → {save_path}")
    else:
        plt.show()

print("Functions ready.")

Functions ready.


In [ ]:
# ── Load both models for comparison
from src.gradcam import load_model_from_experiment, get_target_layer, collect_samples
from src.cbam import CBAMEfficientNetB3
from src.dataset import BrainTumorDataset
from src.transforms import get_transforms
from torch.utils.data import DataLoader

device      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_DIR    = '/content/drive/MyDrive/brain_tumor_classification/data'
EXP_DIR     = '/content/drive/MyDrive/brain_tumor_classification/experiments'
OUTPUTS_DIR = '/content/drive/MyDrive/brain_tumor_classification/outputs/gradcam'
os.makedirs(OUTPUTS_DIR, exist_ok=True)

# Load baseline EfficientNetB3
print("Loading baseline EfficientNetB3...")
model_base, _ = load_model_from_experiment(
    'baseline_efficientnet_b3', device)
layer_base = get_target_layer(model_base, 'efficientnet_b3')
print("  Baseline loaded.")

# Load CBAM model
print("Loading CBAM EfficientNetB3...")
model_cbam = CBAMEfficientNetB3(
    num_classes=4, dropout_rate=0.3, pretrained=False)
model_cbam.load_state_dict(torch.load(
    os.path.join(EXP_DIR, 'cbam_efficientnet_b3', 'best_model.pth'),
    map_location=device))
model_cbam = model_cbam.to(device)
model_cbam.eval()
layer_cbam = model_cbam.backbone.blocks[-1]
print("  CBAM model loaded.")

# Build test loader
test_loader = DataLoader(
    BrainTumorDataset(
        root_dir  = os.path.join(DATA_DIR, 'Testing'),
        transform = get_transforms(224, 'test'),
    ),
    batch_size=16, shuffle=True, num_workers=2
)

# Collect 4 misclassified samples from baseline
print("\nCollecting misclassified samples...")
_, incorrect = collect_samples(
    model_base, test_loader, device,
    n_correct=0, n_incorrect=4
)
print(f"  Collected {len(incorrect)} misclassified samples.")

Loading baseline EfficientNetB3...
  Baseline loaded.
Loading CBAM EfficientNetB3...
  CBAM model loaded.

  Collected 4 misclassified samples.


In [ ]:
# ── Generate Grad-CAM comparison figure
plot_comparison_fixed(
    model_a   = model_base,
    layer_a   = layer_base,
    model_b   = model_cbam,
    layer_b   = layer_cbam,
    samples   = incorrect,
    label_a   = "EfficientNetB3 (baseline)",
    label_b   = "EfficientNetB3 + CBAM",
    device    = device,
    save_path = os.path.join(OUTPUTS_DIR, 'gradcam_baseline_vs_cbam.png'),
)

Saved → /content/drive/MyDrive/brain_tumor_classification/outputs/gradcam/gradcam_baseline_vs_cbam.png
